# 🔬 Phase 4 Lock Diagnostics
Run cells **1–4** while the lock is running. Cell 5 stops the lock and rescans — run it last if needed.

In [2]:
import sys, pathlib

RP_IP    = "192.168.0.99"
RP_KEY   = "Cav"
SSH_USER = "root"
SSH_PASS = "root"

# Scan params
CAV_DEC       = 32                            # change scan period
CAV_AMP       = 0.38                           # V — scan amplitude
CAV_OFFSET    = 0.4                           # V — scan offset
CAV_RANGE     = [[2.47, 2.53], [2.88, 2.97]] # ms — reference peak windows
CAV_LOCKPOINT = 2.93                          # ms — lockpoint
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

SHOW_TRIGGER  = True

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

# Locate repo root
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

print(f"Board     : {RP_IP}")
print(f"Scan      : amp={CAV_AMP}V  offset={CAV_OFFSET}V  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms")
print(f"Lock      : range={CAV_RANGE}  lp={CAV_LOCKPOINT}ms")

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL
Board     : 192.168.0.99
Scan      : amp=0.38V  offset=0.4V  dec=32  period=4.194ms
Lock      : range=[[2.47, 2.53], [2.88, 2.97]]  lp=2.93ms


In [3]:
import threading, time
from lockclient import LockClient, RP_client, Monitor

Monitor.show_trigger = SHOW_TRIGGER

# Clean up any stale session
try:
    if "Lock" in dir() and Lock is not None:
        try: Lock.close()
        except: pass
        # Reset stale lsock on all RPs
        for rp in Lock.RPs.values():
            rp.lsock = None
            rp.loop_running = False
    time.sleep(1)
except: pass

RPs = {"Cav": RP_client((RP_IP, 5000), {}, mode="scan_mon")}

print("Uploading and connecting...")
Lock = LockClient(RPs)

err = {}
def _connect():
    try: Lock.connect_all()
    except Exception as e: err["e"] = e

t = threading.Thread(target=_connect, daemon=True)
t.start(); t.join(timeout=45)
if t.is_alive():   raise TimeoutError("connect_all timed out")
if "e" in err:     raise RuntimeError(f"connect_all failed: {err['e']}")
print("Connected OK")

stcl_thread = threading.Thread(target=Lock.start, daemon=True)
stcl_thread.start()
time.sleep(2)

Lock.set_dec("Cav", CAV_DEC)
print(f"Event loop started  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms")

Uploading and connecting...
connecting...
Connected OK
Event loop started  dec=32  period=4.194ms


In [4]:
# DIAG 1 — PC-side state snapshot

rp = Lock.RPs["Cav"]
s  = Lock.RPs["Cav"].settings["Master"]

print("=" * 55)
print("DIAG 1 — PC-side state")
print("=" * 55)
print(f"  loop_running   : {rp.loop_running}")
print(f"  lsock          : {rp.lsock}")
print(f"  mode           : {rp.mode}")
print(f"  dec            : {s.get('dec')}")
print(f"  range          : {s.get('range')}")
print(f"  lockpoint      : {s.get('lockpoint')} ms")
print(f"  enabled        : {s.get('enabled')}")
pid = s.get("PID", {})
print(f"  PID P/I/D      : {pid.get('P')} / {pid.get('I')} / {pid.get('D')}")
print(f"  PID I_val      : {pid.get('I_val')}")
print(f"  PID limit      : {pid.get('limit')}")
print(f"  monitor running: {Lock.monitors['Cav']['running'].value}")

DIAG 1 — PC-side state
  loop_running   : False
  lsock          : None
  mode           : scan_mon
  dec            : 32
  range          : [[1.9, 2.09], [2.4, 2.5]]
  lockpoint      : 2.47 ms
  enabled        : True
  PID P/I/D      : 0.0 / 2.0 / 0.0
  PID I_val      : 0
  PID limit      : [-0.99, 0.99]
  monitor running: 0


In [5]:
# DIAG 2 — Live port 5066 data (what monitor is actually plotting)
def fetch_5066(ch=0, ip=Lock.RPs["Cav"].addr[0]):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(3)
    s.connect((ip, 5066))
    s.sendall(bytes([ch]))
    hdr = b""
    while len(hdr) < 4:
        hdr += s.recv(4 - len(hdr))
    length = struct.unpack(">I", hdr)[0]
    payload = b""
    while len(payload) < length:
        payload += s.recv(min(65536, length - len(payload)))
    s.close()
    dur, data = json.loads(payload.decode())
    return dur, np.array(data)

print("=" * 55)
print("DIAG 2 — Live port 5066 data (3 snapshots, 0.5s apart)")
print("=" * 55)
snaps = []
for i in range(3):
    try:
        dur, data = fetch_5066(0)
        snaps.append(data.copy())
        print(f"  [{i+1}] dur={dur:.3f}ms  N={len(data)}"
              f"  min={data.min():.4f}V  max={data.max():.4f}V"
              f"  mean={data.mean():.4f}V  std={data.std():.4f}V")
    except Exception as e:
        print(f"  [{i+1}] FAILED: {e}")
    time.sleep(0.5)

if len(snaps) >= 2:
    if np.allclose(snaps[0], snaps[-1]):
        print("\n  ⚠ WARNING: snapshots IDENTICAL — ADC cache frozen during lock")
        print("    Fix: check _adc_cache update in RP_Lock.step()")
    else:
        print("\n  ✓ Snapshots differ — cache is updating live")

DIAG 2 — Live port 5066 data (3 snapshots, 0.5s apart)
  [1] dur=4.194ms  N=16384  min=-0.0415V  max=-0.0073V  mean=-0.0231V  std=0.0042V
  [2] dur=4.194ms  N=16384  min=-0.0415V  max=-0.0073V  mean=-0.0231V  std=0.0042V
  [3] dur=4.194ms  N=16384  min=-0.0415V  max=-0.0073V  mean=-0.0231V  std=0.0042V

  ⚠ WARNING: snapshots IDENTICAL — ADC cache frozen during lock
    Fix: check _adc_cache update in RP_Lock.step()


In [6]:
# DIAG 3 — Board state via SSH
import paramiko

RP_IP = Lock.RPs["Cav"].addr[0]
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username="root", password="root")

print("=" * 55)
print("DIAG 3 — Board state")
print("=" * 55)

_, out, _ = ssh.exec_command("ss -tlnp | grep -E '5000|5065|5066'")
ports = out.read().decode().strip()
print("  Open ports:")
for line in (ports.splitlines() if ports else ["  (none — RunLock.py may have crashed!)"] ):
    print("   ", line)

_, out, _ = ssh.exec_command("ps aux | grep RunLock | grep -v grep")
proc = out.read().decode().strip()
print(f"\n  RunLock.py: {proc if proc else 'NOT RUNNING ✗'}")

# Read gen_ramp offset (PID output) directly from board
_, out, err = ssh.exec_command(
    "python3 -c 'import rp; rp.rp_Init(); "
    "r,v=rp.rp_GenGetOffset(rp.RP_CH_2); print(f\"Out2 offset={v:.4f}V\")' 2>&1"
)
print(f"\n  {out.read().decode().strip()}")

ssh.close()

DIAG 3 — Board state
  Open ports:
    LISTEN 0      128     192.168.0.99:5000      0.0.0.0:*    users:(("python3",pid=937,fd=7))                       
    LISTEN 0      128     192.168.0.99:5065      0.0.0.0:*    users:(("python3",pid=937,fd=10))                      
    LISTEN 0      5            0.0.0.0:5066      0.0.0.0:*    users:(("python3",pid=937,fd=8))

  RunLock.py: root       937 37.6  4.5 131280 21592 ?        Ssl  07:51  20:14 python3 /root/RunLock.py

  Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'rp'


In [ ]:
# DIAG 4 — IN1 mean trend: is PID correcting or saturating?
print("=" * 55)
print("DIAG 4 — IN1 mean over 10 snapshots")
print("  (stable mean = locked, drifting = not locked, frozen = stale cache)")
print("=" * 55)
means, stds = [], []
for i in range(10):
    try:
        _, data = fetch_5066(0)
        means.append(float(data.mean()))
        stds.append(float(data.std()))
    except Exception as e:
        means.append(float('nan'))
        stds.append(float('nan'))
    time.sleep(0.3)

for i, (m, sd) in enumerate(zip(means, stds)):
    bar = "█" * int(abs(m) * 200)
    print(f"  t={i*0.3:.1f}s  mean={m:+.4f}V  std={sd:.4f}V  {bar}")

mean_arr = np.array(means)
drift = float(np.nanmax(mean_arr) - np.nanmin(mean_arr))
frozen = bool(np.all(np.isclose(mean_arr, mean_arr[0], atol=1e-6)))
print(f"\n  Peak-to-peak drift: {drift:.4f} V")
if frozen:
    print("  ⚠ FROZEN — ADC cache not updating during lock mode")
    print("    → RP_Lock.step() is not writing to self._adc_cache")
elif drift < 0.003:
    print("  ✓ Very stable — cavity is likely locked")
elif drift < 0.015:
    print("  ~ Slow drift — PID correcting, or lock is loose")
    print("    → Try increasing I gain slightly")
else:
    print("  ✗ Large drift — PID not holding lock")
    print("    → Check PID sign, gain, and lockpoint position")

## DIAG 5 — Re-scan (stops lock)
Only run if DIAG 4 shows the lock is not holding.

In [ ]:
# DIAG 5 — Re-scan to verify peaks (STOPS THE LOCK — run last)
# Uncomment to run.
# Lock.stop_loop("Cav")
# time.sleep(2)
# Lock.start_scan("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)
# time.sleep(3)
# _, data = fetch_5066(0)
# import matplotlib.pyplot as plt
# t = np.linspace(0, Lock.RPs["Cav"].settings["Master"]["dec"] * 8e-9 * 16384 * 1e3, len(data))
# plt.figure(figsize=(11, 3))
# plt.plot(t, data, lw=0.5, color="#4fc3f7")
# plt.axvline(CAV_LOCKPOINT, color="orange", ls="--", lw=1.5, label=f"lockpoint {CAV_LOCKPOINT}ms")
# for r in CAV_RANGE:
#     plt.axvspan(r[0], r[1], alpha=0.15, color="royalblue")
# plt.xlabel("Time [ms]"); plt.ylabel("IN1 [V]")
# plt.title("DIAG 5 — Scan sweep: peaks must be inside blue ranges")
# plt.legend(); plt.tight_layout(); plt.show()
# print("Re-run Phase 4 cell after tuning.")